# A2: JupyterHub probe 512×512 (in-kernel)

Thử suy luận InstructPix2Pix trên validation, **không** phải fine-tune mới, không chọn tham số bằng test cũ.
Chạy sau khi A1 xong hoặc trên GPU khác. Không dùng `subprocess`: pipeline + CLIP nạp **một lần** rồi dùng lại cho mọi cấu hình guidance.

## 1. Cấu hình

In [ ]:
from pathlib import Path
import os, sys

# Notebook nằm ở A2/; nếu mở chỗ khác thì sửa A2_DIR.
A2_DIR = Path.cwd() if (Path.cwd() / 'instructpix2pix' / 'compare.py').is_file() else Path.home() / 'SDH_HK261_CO5085' / 'A2'
DATA = Path.home() / 'datasets' / 'ip2p_probe'   # split 80/10/10 tạo bởi prepare_data
CHECKPOINT = None        # ví dụ A2_DIR / 'runs/ip2p_v1/checkpoint-2000' (folder chứa unet/)
RUN_SWEEP = False        # True: text {7.5, 10} × image {1.2, 1.5, 1.8}
LIMIT, RESOLUTION, STEPS = 20, 512, 30
OUT = A2_DIR / 'runs' / ('probe512_ft' if CHECKPOINT else 'probe512')

IP2P = A2_DIR / 'instructpix2pix'
assert (IP2P / 'compare.py').is_file(), f'Không thấy code A2 tại {A2_DIR}'
if str(IP2P) not in sys.path:
    sys.path.insert(0, str(IP2P))
print('A2:', A2_DIR, '| OUT:', OUT)

## 2. Môi trường
Chỉ cài khi thiếu. Nếu kernel đã import `transformers`/`diffusers` phiên bản khác (ví dụ vừa chạy A1 cùng kernel), restart kernel sau khi cài.

In [ ]:
import importlib.util
if any(importlib.util.find_spec(m) is None for m in ['diffusers', 'transformers', 'datasets', 'accelerate']):
    %pip install -q -r {IP2P / 'requirements.txt'}
import torch
assert torch.cuda.is_available(), 'Chọn kernel/profile có GPU'
print('torch', torch.__version__, '|', torch.cuda.get_device_name(0),
      f'| {torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GiB')

## 3. Dữ liệu

In [ ]:
import prepare_data
if not DATA.exists():
    prepare_data.main(['--output', str(DATA)])
print((DATA / 'split_manifest.json').read_text()[:300] if (DATA / 'split_manifest.json').exists() else DATA)

## 4. Nạp model một lần

In [ ]:
import importlib, compare
importlib.reload(compare)
from datasets import load_from_disk
models = compare.load_models(unet=str(CHECKPOINT) if CHECKPOINT else None)
val = load_from_disk(str(DATA))['validation']
print('validation:', len(val), '| VRAM đang dùng:', f'{torch.cuda.memory_allocated() / 2**30:.1f} GiB')

## 5. Chạy probe
Bỏ qua cấu hình đã có `summary.json`. Interrupt an toàn; folder dở dang cần xóa trước khi chạy lại.

In [ ]:
import json
configs = [(t, i) for t in [7.5, 10.] for i in [1.2, 1.5, 1.8]] if RUN_SWEEP else [(7.5, 1.5)]
for text_scale, image_scale in configs:
    output = OUT / f't{text_scale:g}_i{image_scale:g}'
    if (output / 'summary.json').exists():
        print('đã có', output.name); continue
    argv = ['--data', str(DATA), '--output', str(output), '--split', 'validation', '--limit', str(LIMIT),
            '--resolution', str(RESOLUTION), '--steps', str(STEPS),
            '--text_guidance', str(text_scale), '--image_guidance', str(image_scale)]
    if CHECKPOINT:
        argv += ['--unet', str(CHECKPOINT)]
    s = compare.evaluate(compare.parse_args(argv), models=models, dataset=val, verbose=False)
    print(f"{output.name}: CLIP={s['mean_clip_target_similarity']:.4f} MAE={s['mean_paired_mae']:.4f} "
          f"{s['mean_seconds']:.2f}s/ảnh flagged={s['safety_flagged']}")

## 6. Xem kết quả

In [ ]:
import pandas as pd
from IPython.display import display, Image
rows = []
for f in sorted(OUT.glob('*/summary.json')):
    s = json.loads(f.read_text())
    rows.append(dict(config=f.parent.name, n=s['count'], clip=s['mean_clip_target_similarity'],
                     mae=s['mean_paired_mae'], sec=s['mean_seconds'], flagged=s['safety_flagged']))
display(pd.DataFrame(rows).sort_values('clip', ascending=False))
SHOW = 3   # số ảnh input | target | prediction mỗi cấu hình
for d in sorted(p for p in OUT.iterdir() if p.is_dir()):
    print(d.name)
    for p in sorted(d.glob('*_input_target_prediction.png'))[:SHOW]:
        display(Image(filename=str(p), width=768))

## 7. Xuất kết quả nhỏ

In [ ]:
import zipfile
from IPython.display import FileLink
EXPORT = A2_DIR / 'exports'; EXPORT.mkdir(exist_ok=True)
archive = EXPORT / f'{OUT.name}.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob('*'):
        if p.suffix in {'.json', '.csv'} or p.name.endswith('_input_target_prediction.png'):
            z.write(p, p.relative_to(A2_DIR))
print(f'{archive} ({archive.stat().st_size / 2**20:.1f} MB)')
display(FileLink(os.path.relpath(archive, Path.cwd())))

## Bước tiếp theo
Xem `IMPROVEMENT_PLAN.md`. Chưa có loader HQ-Edit hay fine-tune 512 mới. Baseline ip2p_v1 giữ nguyên trong `REPORT.md`.